In [1]:
import pandas as pd

In [2]:
filepath = r"C:\Users\HP\Desktop\Projects - Data Analysis\RSF\CKM_Dummy_EHR_Dataset.xlsx"
total_data = pd.ExcelFile(path_or_buffer= filepath)

sheet_dict = dict()
for sheetname in total_data.sheet_names:
    sheet_dict[sheetname] = total_data.parse(sheet_name= sheetname)

In [3]:
(patient_data, encounter_data, vitals_data, lab_test_data, diagnosis_data, 
 medications_data, history_data, procedures_data) = sheet_dict.values()

In [4]:
df_patient_data = patient_data.drop(columns=[
    'insurance_category', 'cause_of_death'])
df_enctr_data = encounter_data.drop(columns=[
    'facility_code', 'facility_name', 'encounter_type', 'discharge_status'
    ])
df_vitals_data = vitals_data.drop(columns=[
    'height_unit', 'weight_unit', 'bmi_unit', 'waist_circumference_unit', 
    'source_facility_code'])
df_lab_test_data = lab_test_data.drop(columns=[
    'result_value_text', 'reference_low', 'reference_high', 'result_status', 
    'facility_code'])
df_diagnosis_data = diagnosis_data.drop(columns=[
    'diagnosis_type', 'diagnosis_source', 'active_status'])
df_medications_data = medications_data.drop(columns=[
    'medication_record_id', 'generic_name', 'medication_code', 'frequency', 
    'route', 'medication_status', 'order_date', 'start_date', 'end_date'])                                                
df_history_data = history_data.drop(columns=[
    'recorded_date', 'data_source'])
df_procedures_data = procedures_data.drop(columns=[
    'event_date', 'event_category', 'description', 'status', 'facility_code', 
    'result_unit'])

Check for Duplicates

In [5]:
dfs = [df_patient_data, df_enctr_data, df_vitals_data, df_lab_test_data, 
       df_diagnosis_data, df_medications_data, df_history_data, 
       df_procedures_data]

for df in dfs:
    if df.duplicated().any():
        print(f"\nThere are duplicates.")
    else:
        print(f"\nThere are no duplicates.")


There are no duplicates.

There are no duplicates.

There are no duplicates.

There are no duplicates.

There are no duplicates.

There are no duplicates.

There are no duplicates.

There are no duplicates.


Pivot Tables

In [6]:
pivot_lab = pd.pivot_table(lab_test_data, values= 'result_value_numeric', 
                            index= ['patient_id', 'encounter_id'],
                            columns= 'test_name')

In [7]:
pivot_icd =pd.pivot_table(
    df_diagnosis_data, values= 'icd10_code', 
    index= ['patient_id', 'encounter_id'], columns= 'diagnosis_description', 
    aggfunc= 'any'
    )
pivot_icd_v2 = (pivot_icd.replace(to_replace=[True, False], 
                                  value= [int(1), int(0)])).fillna(0)
pivot_icd_v2

diagnosis_description   Acute myocardial infarction, unspecified  \
patient_id encounter_id                                            
PT000001   ENC00000003                                         0   
           ENC00000016                                         0   
PT000002   ENC00000039                                         0   
           ENC00000048                                         0   
PT000003   ENC00000084                                         0   
...                                                          ...   
PT001194   ENC00028524                                         0   
           ENC00028548                                         1   
PT001196   ENC00028576                                         0   
PT001197   ENC00028584                                         0   
PT001199   ENC00028628                                         0   

diagnosis_description   Atherosclerotic heart disease, subclinical  \
patient_id encounter_id                                              
PT000001   ENC00000003                                           0   
           ENC00000016                                           0   
PT000002   ENC00000039                                           0   
           ENC00000048                                           0   
PT000003   ENC00000084                                           0   
...                                                            ...   
PT001194   ENC00028524                                           1   
           ENC00028548                                           0   
PT001196   ENC00028576                                           0   
PT001197   ENC00028584                                           0   
PT001199   ENC00028628                                           1   

diagnosis_description   Cerebral infarction, unspecified  \
patient_id encounter_id                                    
PT000001   ENC00000003                                 0   
           ENC00000016                                 0   
PT000002   ENC00000039                                 0   
           ENC00000048                                 0   
PT000003   ENC00000084                                 0   
...                                                  ...   
PT001194   ENC00028524                                 0   
           ENC00028548                                 1   
PT001196   ENC00028576                                 0   
PT001197   ENC00028584                                 0   
PT001199   ENC00028628                                 0   

diagnosis_description   Chronic kidney disease, stage 3  \
patient_id encounter_id                                   
PT000001   ENC00000003                                0   
           ENC00000016                                1   
PT000002   ENC00000039                                0   
           ENC00000048                                1   
PT000003   ENC00000084                                0   
...                                                 ...   
PT001194   ENC00028524                                0   
           ENC00028548                                0   
PT001196   ENC00028576                                0   
PT001197   ENC00028584                                0   
PT001199   ENC00028628                                1   

diagnosis_description   Chronic kidney disease, stage 4  \
patient_id encounter_id                                   
PT000001   ENC00000003                                0   
           ENC00000016                                0   
PT000002   ENC00000039                                0   
           ENC00000048                                0   
PT000003   ENC00000084                                0   
...                                                 ...   
PT001194   ENC00028524                                1   
           ENC00028548                                0   
PT001196   ENC00028576                                0   
PT0011

In [8]:
history = pd.get_dummies(
    df_history_data[['smoking_status', 'alcohol_use']], drop_first=True
    )
history_ready = pd.concat(
    [df_history_data[['patient_id', 'encounter_id']], history], axis= 1
    )
history_ready = (history_ready.replace(to_replace=[True, False],
                                        value=[int(1), int(0)])).fillna(0)

In [ ]:
'''pivot_smoking = pd.pivot_table(df_history_data.drop(columns=['alcohol_use']), values= 'smoking_status',index=['patient_id', 'encounter_id'], columns= ['smoking_status'], aggfunc='any')
pivot_smoking = (pivot_smoking.replace(to_replace=True, value= int(1))).fillna(0)

pivot_alcohol = pd.pivot_table(df_history_data.drop(columns=['smoking_status']), values= 'alcohol_use', index= ['patient_id', 'encounter_id'], columns= ['alcohol_use'], aggfunc='any')
pivot_alcohol = (pivot_alcohol.replace(to_replace=True, value= int(1))).fillna(0)'''

Flatten Tables

In [9]:
flat_lab_data = pivot_lab.reset_index()

flat_icd = pivot_icd_v2.reset_index()

Merging Tables

In [10]:
frames = [df_vitals_data, flat_lab_data, flat_icd, #df_medications_data, 
          history_ready]

merged_df = pd.merge(df_patient_data, df_enctr_data, on= 'patient_id', 
                     how= 'outer')
for frame in frames:
    merged_df = pd.merge(merged_df, frame, 
                         on= ['patient_id', 'encounter_id'], how= 'outer')

Sorting via .groupby()

In [11]:
merged_df['encounter_datetime'] = pd.to_datetime(merged_df['encounter_datetime'])
data_sorted = merged_df.sort_values(by='encounter_datetime')
data_sorted

,patient_id,sex,year_of_birth,nationality,death_date,encounter_id,encounter_datetime,discharge_datetime,department_specialty,measurement_datetime,...,"Peripheral artery disease, unspecified extremity",Prediabetes,"Proteinuria, severely increased",Type 2 diabetes mellitus without complications,smoking_status_Former,smoking_status_Never,smoking_status_Unknown,alcohol_use_Occasional,alcohol_use_Regular,alcohol_use_Unknown
26248,PT001099,M,1938,Egypt,NaT,ENC00026249,2015-01-01 07:00:00,2015-01-01 09:00:00,Nephrology,2015-01-01 07:00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11090,PT000454,F,1934,India,NaT,ENC00011091,2015-01-01 07:00:00,2015-01-02 07:00:00,Emergency Medicine,2015-01-01 07:00:00,...,0,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN
18265,PT000758,F,1934,United Kingdom,NaT,ENC00018266,2015-01-01 10:00:00,2015-01-01 10:00:00,Endocrinology,2015-01-01 10:00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12292,PT000502,F,1983,Philippines,NaT,ENC00012293,2015-01-01 10:00:00,2015-01-01 10:00:00,Endocrinology,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21907,PT000910,F,1974,India,2022-11-07,ENC00021908,2015-01-01 13:00:00,2015-01-01 13:00:00,Cardiology,2015-01-01 13:00:00,...,NaN,NaN,NaN,NaN,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20304,PT000843,F,1975,India,NaT,ENC00020305,2025-12-31 13:00:00,2026-01-05 13:00:00,Cardiology,2025-12-31 13:00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2127,PT000081,F,1987,UAE,NaT,ENC00002128,2025-12-31 13:00:00,2025-12-31 13:00:00,Family Medicine,2025-12-31 13:00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18801,PT000782,M,1957,India,NaT,ENC00018802,2025-12-31 17:00:00,2025-12-31 17:00:00,Endocrinology,2025-12-31 17:00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
26989,PT001125,F,1939,India,NaT,ENC00026990,2025-12-31 18:00:00,2025-12-31 18:00:00,Family Medicine,2025-12-31 18:00:00,...,NaN,NaN,NaN,NaN,0,1,0,1,0,0


Encode Data

In [12]:
# TRYING OUT HOT ENCODING AND BASE INDEX:

test_grouped_data = data_sorted.groupby('patient_id').first(skipna=False)

test_encoded_data = pd.get_dummies(
    test_grouped_data[['sex', 'nationality']], drop_first=True
    )

hot_encoded_data = pd.concat([test_grouped_data.drop(
    columns=['sex', 'nationality']),test_encoded_data], axis= 1
    )

# Make this a for col in list thingy to fix Nan issue?
for col in ['sex_M', 'nationality_Egypt',
       'nationality_India', 'nationality_Jordan', 'nationality_Other',
       'nationality_Pakistan', 'nationality_Philippines', 'nationality_UAE',
       'nationality_United Kingdom']:
    hot_encoded_data[col] = hot_encoded_data[col].replace(
        to_replace=True, value= int(1)).fillna(int(0))

date_columns = ['encounter_datetime', 'discharge_datetime', 
                'death_date', 'year_of_birth']
for col in date_columns:
    hot_encoded_data[col] = pd.to_datetime(hot_encoded_data[col])
    hot_encoded_data[col] = hot_encoded_data[col].apply(lambda x: x.year)

for col in hot_encoded_data.select_dtypes(['object']):
    if col not in ['encounter_id', 'department_specialty', 'medication_name', 
                   'medication_class', 'dose_unit']:
        hot_encoded_data[col] = hot_encoded_data[col].apply(
            lambda x: int(x) if pd.notnull(x) else x
            )
        hot_encoded_data[col] = hot_encoded_data[col].fillna(int(0))
hot_encoded_data['death_date'] = hot_encoded_data['death_date'].apply(lambda x: 1 if pd.notnull(x) else 0)


C:\Users\HP\AppData\Local\Temp\ipykernel_20808\1956993056.py:27: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in hot_encoded_data.select_dtypes(['object']):


In [13]:
re_hot_encoded_data = hot_encoded_data.drop(
    columns=['department_specialty', 'measurement_datetime'])

In [ ]:
re_hot_encoded_data.to_csv('hot_encoded_ckm_grouped_by_encounter_date.csv') 

In [ ]:
'''from sklearn.preprocessing import LabelEncoder

encoded_data = grouped_data.copy()

for column in encoded_data.columns:
    if pd.api.types.is_string_dtype(encoded_data[column]) and column != ['encounter_date', 'discharge_date', 'death_date']:
        label_encode = LabelEncoder()
        encoded_data[column] = label_encode.fit_transform(encoded_data[column])'''


In [ ]:
'''date_columns = ['encounter_date', 'discharge_date', 'death_date', 'dob']
for col in date_columns:
    encoded_data[col] = pd.to_datetime(encoded_data[col])
    encoded_data[col] = encoded_data[col].apply(lambda x: x.year)
    var_dict = dict()
    i=1
    for var in encoded_data[col].unique():
        var_dict[var] = int(i)
        i += 1
    print(var_dict)
    encoded_data[col] = encoded_data[col].replace(var_dict)'''

In [ ]:
#encoded_data.to_csv('encoded_ckm_grouped_by_encounter_date.csv', index=False) 